# Análise de Projetos de Investimento - Distrito Federal

## 1. Introdução e Contexto

### Documentação de Configurações

### Fonte dos Dados

Os dados utilizados nesta análise foram extraídos da **API ObrasGov.br**, especificamente do endpoint `/projeto-investimento`.

- **URL Base da API:** `https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento`
- **Filtro Aplicado:** A extração foi restrita aos projetos localizados na **Unidade Federativa (UF) do Distrito Federal (DF)**, utilizando o parâmetro `uf=DF`.
- **Formato:** Os dados são retornados em formato JSON, com estrutura aninhada (listas de dicionários) para campos como executores, fontes de recurso e tipos de projeto.

### Objetivo da Análise

O objetivo principal desta análise é demonstrar a capacidade de construir um pipeline de processamento de dados (ETL - Extração, Transformação e Carga) e realizar uma análise exploratória sobre dados públicos de projetos de investimento.

**Objetivos Específicos:**

1. Integrar-se com uma API pública (ObrasGov.br) para extrair dados de forma paginada.
2. Tratar e normalizar dados complexos.
3. Persistir os dados tratados em um banco de dados relacional (PostgreSQL).
4. Gerar insights e visualizações que respondam a perguntas de negócio sobre os investimentos no DF.




## 2. Extração de dados
### 2.1 Importações 


In [41]:
# Importar bibliotecas necessárias
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time
from datetime import datetime
import json
from tqdm import tqdm

### 2.2 Definição de constantes e funções

In [86]:
url = "https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento"
pagina_atual = 0
registros_por_pagina = 100
uf_filtro = "DF"
lista_de_obras = []

print("Iniciando extração de dados da API ObrasGov.br (UF=DF)")

while True: 
    print(f"Extraindo dados da página {pagina_atual}...", end='\r')
    
    params = {
        "pagina": pagina_atual, 
        "tamanhoDaPagina": registros_por_pagina,
        "uf": uf_filtro,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        content = response.json().get('content', [])
        
        if content: 
            lista_de_obras.extend(content)
            pagina_atual += 1
        else: 
            print("\nExtração de dados finalizada.")
            break

    except requests.exceptions.RequestException as e:
        print(f"\nOcorreu um erro na requisição. {e}")
        break


Iniciando extração de dados da API ObrasGov.br (UF=DF)
Extraindo dados da página 9...
Extração de dados finalizada.


### 2.3 Extração e salvamento dos dados

In [43]:
import os  
import json 
# --- Salvamento dos Dados ---
# A variável \'lista_de_obras\' já contém os dados extraídos da célula anterior
dados_brutos = lista_de_obras

# Cria o diretório para salvar os dados brutos, se não existir
os.makedirs("data/raw", exist_ok=True)

# Salvar dados brutos
if dados_brutos:
    raw_file_path = 'data/raw/projetos_df_raw.json'
    with open(raw_file_path, 'w', encoding='utf-8') as f:
        json.dump(dados_brutos, f, ensure_ascii=False, indent=4)
    print(f"\nDados brutos salvos em: {raw_file_path}")
    print(f"Total de registros: {len(dados_brutos)}")
    
    # Mostrar amostra dos dados
    print("\nAmostra do primeiro registro:")
    # Usamos [0] para pegar o primeiro elemento da lista de projetos
    print(json.dumps(dados_brutos[0], indent=2)) 
else:
    print("Nenhum dado para salvar.")


Dados brutos salvos em: data/raw/projetos_df_raw.json
Total de registros: 834

Amostra do primeiro registro:
{
  "idUnico": "50379.53-54",
  "nome": "DL - 304/2024 - Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "cep": null,
  "endereco": null,
  "descricao": "Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "funcaoSocial": "Amplia\u00e7\u00e3o da capacidade de trafego visando a melhoria da seguran\u00e7a do usu\u00e1rio",
  "metaGlobal": "Projetos B\u00e1sicos e Executivos de Engenharia",
  "dataInicialPrevista": "2024-12-20",
  "dataFinalPrevista": "2027-12-05",
  "dataInicialEfetiva": null,
  "dataFinalE

# 3. Análise Exploratória
### 3.1 - Carregar Dados em DataFrame
Objetivo: Converter a lista de dicionários extraída da API (dados_brutos) em um objeto DataFrame do Pandas para manipulação e análise.




In [44]:
# Tenta carregar a variável 'dados_brutos' da memória, ou recarrega do arquivo JSON
try:
    # Verifica se a variável 'dados_brutos' existe e não está vazia
    if 'dados_brutos' not in locals() or not dados_brutos:
        with open('data/raw/projetos_df_raw.json', 'r', encoding='utf-8') as f:
            dados_brutos = json.load(f)
            print("Dados brutos recarregados do arquivo JSON.")
    
    # Cria o DataFrame
    df = pd.DataFrame(dados_brutos)
    print(f"DataFrame criado com sucesso. Dimensões iniciais: {df.shape[0]} linhas e {df.shape[1]} colunas.")
    
except FileNotFoundError:
    print("ERRO: O arquivo 'data/raw/projetos_df_raw.json' não foi encontrado. Execute a extração (Passo 3) novamente.")
    df = pd.DataFrame() # Cria um DataFrame vazio para evitar erros

DataFrame criado com sucesso. Dimensões iniciais: 834 linhas e 31 colunas.


### 3.2 Análise Inicial

Objetivo: Obter uma visão geral da estrutura do DataFrame, tipos de dados e estatísticas descritivas.



In [45]:
if not df.empty:
    # 1. Dimensões do DataFrame
    print("--- 1. Dimensões do DataFrame ---")
    print(f"O DataFrame possui {df.shape[0]} registros (projetos) e {df.shape[1]} atributos (colunas).")

    # 2. Tipos de Dados e Contagem de Não-Nulos
    print("\n--- 2. Tipos de Dados e Contagem de Não-Nulos (df.info()) ---")
    df.info()

    # 3. Amostra das Primeiras Linhas
    print("\n--- 3. Amostra das Primeiras Linhas (df.head()) ---")
    display(df.head())

    # 4. Lista de Colunas
    print("\n--- 4. Lista Completa de Colunas ---")
    print(df.columns.tolist())

    # 5. Estatísticas Descritivas
    print("\n--- 5. Estatísticas Descritivas (df.describe(include='all')) ---")
    # Usamos include='all' para obter estatísticas de colunas numéricas e categóricas
    display(df.describe(include='all').T)

--- 1. Dimensões do DataFrame ---
O DataFrame possui 834 registros (projetos) e 31 atributos (colunas).

--- 2. Tipos de Dados e Contagem de Não-Nulos (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 834 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   idUnico                             834 non-null    object
 1   nome                                834 non-null    object
 2   cep                                 384 non-null    object
 3   endereco                            425 non-null    object
 4   descricao                           834 non-null    object
 5   funcaoSocial                        834 non-null    object
 6   metaGlobal                          834 non-null    object
 7   dataInicialPrevista                 831 non-null    object
 8   dataFinalPrevista                   831 non-null    object
 9   dataInicialEfetiva 

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,None,None,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,None,...,None,False,2024-12-20,[],[{'nome': 'DEPARTAMENTO NACIONAL DE INFRAESTRU...,[],"[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 25, 'descricao': 'Rodovia', 'idEixo': 3}]","[{'id': 4, 'descricao': 'Acessos Terrestres', ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
1,42724.53-27,Escola Classe Crixá São Sebastião,None,None,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,None,...,None,False,2025-09-05,[],[{'nome': 'SECRETARIA DE ESTADO DE EDUCACAO DO...,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,"[{'id': 4, 'descricao': 'Social'}]","[{'id': 46, 'descricao': 'Educação', 'idEixo':...","[{'id': 84, 'descricao': 'Educação', 'idTipo':...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70.602-600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,None,...,None,False,2023-02-06,[],[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 1, 'descricao': 'Segurança Pública', '...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,None,None,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,None,...,None,False,2023-08-28,[],[{'nome': 'DEPARTAMENTO DE ESTRADAS DE RODAGEM...,"[{'nome': 'MINISTÉRIO DAS CIDADES', 'codigo': ...","[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 24, 'descricao': 'Infraestrutura Urban...","[{'id': 57, 'descricao': 'Obra de Arte Especia...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",None,None,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,None,...,None,False,2023-08-29,[],"[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'id': 3, 'descricao': 'Econômico'}, {'id': 3...","[{'id': 31, 'descricao': 'Energia', 'idEixo': ...","[{'id': 95, 'descricao': 'Subestação', 'idTipo...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."



--- 4. Lista Completa de Colunas ---
['idUnico', 'nome', 'cep', 'endereco', 'descricao', 'funcaoSocial', 'metaGlobal', 'dataInicialPrevista', 'dataFinalPrevista', 'dataInicialEfetiva', 'dataFinalEfetiva', 'dataCadastro', 'especie', 'natureza', 'naturezaOutras', 'situacao', 'descPlanoNacionalPoliticaVinculado', 'uf', 'qdtEmpregosGerados', 'descPopulacaoBeneficiada', 'populacaoBeneficiada', 'observacoesPertinentes', 'isModeladaPorBim', 'dataSituacao', 'tomadores', 'executores', 'repassadores', 'eixos', 'tipos', 'subTipos', 'fontesDeRecurso']

--- 5. Estatísticas Descritivas (df.describe(include='all')) ---


,count,unique,top,freq
idUnico,834,696,79119.53-46,3
nome,834,650,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,13
cep,384,96,1,95
endereco,425,240,,61
descricao,834,632,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,13
funcaoSocial,834,508,Segurança Pública,45
metaGlobal,834,475,Escola de Educação Infantil Tipo B,68
dataInicialPrevista,831,407,2025-06-01,35
dataFinalPrevista,831,458,2027-06-01,35
dataInicialEfetiva,22,13,2018-07-09,5


### 3.3 - Análise de Qualidade

Objetivo: Quantificar a presença de valores nulos e identificar a existência de registros duplicados.



In [46]:
if not df.empty:
    # 1. Porcentagem de Valores Nulos por Coluna
    print("--- 1. Porcentagem de Valores Nulos por Coluna ---")
    nulos_perc = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    nulos_df = pd.DataFrame({'Nulos (%)': nulos_perc[nulos_perc > 0].round(2)})
    display(nulos_df)

    # 2. Identificação de Duplicatas
    print("\n--- 2. Identificação de Duplicatas ---")
    # A verificação de duplicatas no DataFrame completo falha devido às colunas aninhadas (listas/dicionários).
    # Focamos no identificador único do projeto.
    duplicatas_id = df['idUnico'].duplicated().sum()
    print(f"Total de linhas duplicadas (baseado no identificador 'idUnico'): {duplicatas_id}")
    
    # 3. Verificação de Valores Únicos em Colunas Chave
    print("\n--- 3. Verificação de Valores Únicos em Colunas Chave ---")
    for col in ['situacao', 'uf']:
        print(f"Coluna '{col}': {df[col].nunique()} valores únicos. Amostra: {df[col].unique()}")

--- 1. Porcentagem de Valores Nulos por Coluna ---


,Nulos (%)
dataFinalEfetiva,98.80
dataInicialEfetiva,97.36
observacoesPertinentes,83.45
qdtEmpregosGerados,78.78
populacaoBeneficiada,78.54
descPopulacaoBeneficiada,78.18
naturezaOutras,75.18
descPlanoNacionalPoliticaVinculado,64.63
cep,53.96
endereco,49.04



--- 2. Identificação de Duplicatas ---
Total de linhas duplicadas (baseado no identificador 'idUnico'): 138

--- 3. Verificação de Valores Únicos em Colunas Chave ---
Coluna 'situacao': 6 valores únicos. Amostra: ['Cadastrada' 'Cancelada' 'Em execução' 'Concluída' 'Inativada'
 'Paralisada']
Coluna 'uf': 1 valores únicos. Amostra: ['DF']


## 4. Tratamento de Dados
### 4.1 - Limpar Dados

Objetivo: Achatar as colunas aninhadas, remover duplicatas e normalizar os dados.



In [87]:
# --- Achatar Colunas Aninhadas e Extrair Valores ---

# Função para extrair o nome/descrição principal de uma lista de dicionários
def extract_main_info(list_of_dicts, key='nome'):
    if isinstance(list_of_dicts, list) and list_of_dicts:
        # Junta os nomes/descrições com um separador
        return '; '.join([str(d.get(key, '')) for d in list_of_dicts if isinstance(d, dict)])
    return None # Usar None para permitir o tratamento de nulos posterior

# Função para calcular o valor total previsto
def calculate_total_previsto(list_of_dicts):
    if isinstance(list_of_dicts, list):
        # Soma o valorInvestimentoPrevisto de todas as fontes de recurso
        return sum(d.get('valorInvestimentoPrevisto', 0) for d in list_of_dicts if isinstance(d, dict))
    return 0.0

# Aplicar as funções para criar novas colunas
df['orgao_executor'] = df['executores'].apply(lambda x: extract_main_info(x, key='nome'))
df['fonte_recurso'] = df['fontesDeRecurso'].apply(lambda x: extract_main_info(x, key='origem'))
df['tipo_projeto'] = df['tipos'].apply(lambda x: extract_main_info(x, key='descricao'))
df['valor_total_previsto'] = df['fontesDeRecurso'].apply(calculate_total_previsto)

# --- Seleção e Renomeação de Colunas (CORRIGIDO) ---

# Lista de colunas a serem selecionadas (removendo 'dataUltimaAtualizacao')
cols_to_select = [
    'idUnico', 'nome', 'situacao', 'dataInicialPrevista', 'dataFinalPrevista', 
    'dataInicialEfetiva', 'dataFinalEfetiva', 'dataCadastro',
    'orgao_executor', 'fonte_recurso', 'tipo_projeto', 'valor_total_previsto'
]

# Filtra a lista para incluir apenas as colunas que realmente existem no DataFrame
existing_cols = [col for col in cols_to_select if col in df.columns]

# Selecionar apenas as colunas que serão mantidas
df_clean = df[existing_cols].copy()

# Lista de novos nomes (deve corresponder à lista de colunas selecionadas)
new_names = [
    'id_unico', 'nome_projeto', 'status', 'data_inicio_prevista', 'data_fim_prevista',
    'data_inicio_real', 'data_fim_real', 'data_cadastro',
    'orgao_executor', 'fonte_recurso', 'tipo_projeto', 'valor_total_previsto'
]

# Ajusta a lista de novos nomes para corresponder às colunas que foram realmente selecionadas
df_clean.columns = new_names[:len(df_clean.columns)]

# --- Remover Duplicatas ---
df_clean.drop_duplicates(subset=['id_unico'], inplace=True)
print(f"Duplicatas removidas. Novo tamanho: {df_clean.shape[0]} linhas.")

# --- Normalizar Strings e Tratar Nulos Categóricos ---
# Normalizar strings (remover espaços e converter para maiúsculas/minúsculas)
for col in ['nome_projeto', 'status', 'orgao_executor', 'fonte_recurso', 'tipo_projeto']:
    # Tratar nulos categóricos com 'Não Informado'
    df_clean[col] = df_clean[col].fillna('Não Informado').astype(str).str.strip().str.upper()

# Tratar nulos em colunas que foram achatadas (usamos None na função, agora preenchemos)
df_clean['orgao_executor'] = df_clean['orgao_executor'].replace('NONE', 'NÃO INFORMADO')
df_clean['fonte_recurso'] = df_clean['fonte_recurso'].replace('NONE', 'NÃO INFORMADO')
df_clean['tipo_projeto'] = df_clean['tipo_projeto'].replace('NONE', 'NÃO INFORMADO')

Duplicatas removidas. Novo tamanho: 696 linhas.


### 4.2 - Conversão de Tipos

Objetivo: Garantir que as colunas estejam no formato correto para cálculos e visualizações.



In [ ]:
# --- Converter Datas para Datetime ---
date_cols = [
    'data_inicio_prevista', 'data_fim_prevista', 'data_inicio_real', 
    'data_fim_real', 'data_cadastro'
]
for col in date_cols:
    # errors='coerce' transforma valores inválidos em NaT (Not a Time)
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce', utc=True).dt.tz_localize(None)

# --- Converter Valores Monetários para Float ---
df_clean['valor_total_previsto'] = pd.to_numeric(df_clean['valor_total_previsto'], errors='coerce').fillna(0.0)

# --- Categorizar Variáveis Apropriadas ---
for col in ['status', 'orgao_executor', 'fonte_recurso', 'tipo_projeto']:
    df_clean[col] = df_clean[col].astype('category')

### 4.3 - Feature Engineering

Objetivo: Criar novas colunas que enriquecem a análise.



In [ ]:
# --- Ano/Mês de Início do Projeto (Previsto) ---
df_clean['ano_inicio_previsto'] = df_clean['data_inicio_prevista'].dt.year
df_clean['mes_inicio_previsto'] = df_clean['data_inicio_prevista'].dt.to_period('M')
df_clean['mes_inicio_previsto'] = df_clean['mes_inicio_previsto'].astype(str)
# --- Duração Estimada (em dias) ---
df_clean['duracao_prevista_dias'] = (df_clean['data_fim_prevista'] - df_clean['data_inicio_prevista']).dt.days

# --- Flags de Status (Binárias) ---
df_clean['is_concluido'] = df_clean['status'].str.contains('CONCLUIDA|CONCLUÍDA', na=False)
df_clean['is_cancelado'] = df_clean['status'].str.contains('CANCELADA', na=False)
df_clean['is_em_execucao'] = df_clean['status'].str.contains('EM EXECUÇÃO', na=False)

# --- Faixas de Valores (Binning) ---
bins = [0, 100000, 1000000, 10000000, df_clean['valor_total_previsto'].max() + 1]
labels = ['< 100K', '100K - 1M', '1M - 10M', '> 10M']
df_clean['faixa_valor'] = pd.cut(df_clean['valor_total_previsto'], bins=bins, labels=labels, right=False)

### 4.4 - Validar Dados Tratados

Objetivo: Confirmar que os problemas identificados foram resolvidos.



In [ ]:
# 1. Verificar Ausência de Nulos Críticos
print("--- 1. Verificação de Nulos Críticos ---")
print("Nulos em 'orgao_executor':", df_clean['orgao_executor'].isnull().sum())
print("Nulos em 'valor_total_previsto':", df_clean['valor_total_previsto'].isnull().sum())

# 2. Confirmar Tipos Corretos
print("\n--- 2. Confirmação de Tipos (df.info()) ---")
df_clean.info()

# 3. Estatísticas Pós-Tratamento
print("\n--- 3. Estatísticas Descritivas Pós-Tratamento ---")
display(df_clean[['valor_total_previsto', 'duracao_prevista_dias']].describe())

# 4. Amostra Final
print("\n--- 4. Amostra Final do DataFrame Limpo ---")
display(df_clean.head())

### 4.5 - Salvar Dados Tratados




In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# Salvar CSV processado
processed_file_path = 'data/processed/projetos_df_clean.csv'
df_clean.to_csv(processed_file_path, index=False, encoding='utf-8')
print(f"\nDados tratados salvos em: {processed_file_path}")


Dados tratados salvos em: data/processed/projetos_df_clean.csv


## 5. Armazenamento de Dados 

In [ ]:
# Certifique-se de que estas bibliotecas estão instaladas:
# pip install sqlalchemy python-dotenv psycopg2-binary

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import pandas as pd # Necessário para o DataFrame

# --- 1. Configuração e Conexão ---

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Construir a URL de Conexão para PostgreSQL
# O driver para PostgreSQL é 'postgresql+psycopg2'
db_url = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"

# Criar a Engine de Conexão
engine = create_engine(db_url)

# --- 2. Carga dos Dados ---

# Salvar DataFrame no PostgreSQL
# O método to_sql() faz a inserção
try:
    # if_exists='replace' garante que a tabela seja recriada a cada execução
    df_clean.to_sql('projetos', engine, if_exists='replace', index=False)
    
    print(f"Sucesso: {len(df_clean)} registros inseridos na tabela 'projetos' do PostgreSQL.")

    # --- 3. Verificação (Opcional) ---
    # Verifica se os dados foram realmente inseridos
    with engine.connect() as connection:
        count = connection.execute(text("SELECT COUNT(*) FROM projetos")).scalar_one()
        print(f"Verificação: {count} registros contados na tabela.")
        
except Exception as e:
    print(f"ERRO: Falha ao inserir dados no PostgreSQL. Verifique se o serviço está ativo, as permissões e as credenciais no .env. Erro: {e}")

Sucesso: 696 registros inseridos na tabela 'projetos' do PostgreSQL.
Verificação: 696 registros contados na tabela.


In [ ]:
# Célula de Verificação no Jupyter Notebook

import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Carregar variáveis de ambiente
load_dotenv()

# Construir a URL de Conexão (usando as variáveis do .env)
db_url = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

# Ler os dados da tabela 'projetos'
df_from_db = pd.read_sql('SELECT * FROM projetos', engine)

print(f"Dados lidos do PostgreSQL: {len(df_from_db)} registros.")
display(df_from_db.head())

Dados lidos do PostgreSQL: 696 registros.


,id_unico,nome_projeto,status,data_inicio_prevista,data_fim_prevista,data_inicio_real,data_fim_real,data_cadastro,orgao_executor,fonte_recurso,tipo_projeto,valor_total_previsto,ano_inicio_previsto,mes_inicio_previsto,duracao_prevista_dias,is_concluido,is_cancelado,is_em_execucao,faixa_valor
0,50379.53-54,DL - 304/2024 - CONTRATAÇÃO DE INSTITUIÇÃO PAR...,CADASTRADA,2024-12-20,2027-12-05,NaT,NaT,2024-12-20,DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRA...,FEDERAL,RODOVIA,44463443.00,2024.0,2024-12,1080.0,False,False,False,> 10M
1,42724.53-27,ESCOLA CLASSE CRIXÁ SÃO SEBASTIÃO,CANCELADA,2024-09-02,2028-09-02,NaT,NaT,2024-08-30,SECRETARIA DE ESTADO DE EDUCACAO DO DISTRITO F...,FEDERAL,EDUCAÇÃO,12319519.51,2024.0,2024-09,1461.0,False,True,False,> 10M
2,19970.53-78,REAJUSTE DO CONTRATO 45/2021 - CONTRUÇÃO DO CE...,CADASTRADA,2021-09-14,2024-08-28,NaT,NaT,2023-02-06,CORPO DE BOMBEIROS MILITAR DO DISTRITO FEDERAL,FEDERAL,SEGURANÇA PÚBLICA,1177429.91,2021.0,2021-09,1079.0,False,False,False,1M - 10M
3,24797.53-15,IMPLANTAÇÃO DE PASSARELAS NAS ESTRADAS PARQUE ...,CADASTRADA,2023-08-30,2028-08-30,NaT,NaT,2023-08-28,DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRIT...,FEDERAL,INFRAESTRUTURA URBANA E MOBILIDADE,10800000.00,2023.0,2023-08,1827.0,False,False,False,> 10M
4,24822.53-70,"OBRA DE CONSTRUÇÃO DA CABINE DE MEDIÇÃO, LOCA...",CADASTRADA,2023-09-14,2024-03-14,NaT,NaT,2023-08-29,FUNDACAO UNIVERSIDADE DE BRASILIA,FEDERAL,ENERGIA; ENERGIA,928139.70,2023.0,2023-09,182.0,False,False,False,100K - 1M


### Documentação da Estrutura do Banco de Dados (PostgreSQL)

#### Tabela Criada: `projetos`

A tabela `projetos` foi criada no esquema padrão (`public`) do PostgreSQL, utilizando o método `to_sql()` do Pandas.

#### Colunas e Tipos

As colunas da tabela `projetos` refletem a estrutura final do DataFrame `df_clean` após a transformação e *Feature Engineering*. O Pandas e o SQLAlchemy mapeiam os tipos de dados do Python para os tipos nativos do PostgreSQL.

| Coluna | Tipo de Dado (Pandas) | Tipo de Dado (PostgreSQL) | Descrição |
| :--- | :--- | :--- | :--- |
| `id_unico` | `object` | `TEXT` | Identificador único do projeto (Chave Primária Lógica). |
| `nome_projeto` | `category` | `VARCHAR` | Nome do projeto. |
| `status` | `category` | `VARCHAR` | Status atual do projeto (ex: CADASTRADA, CANCELADA). |
| `data_inicio_prevista` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de início prevista. |
| `data_fim_prevista` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de fim prevista. |
| `data_inicio_real` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de início efetiva (pode ser nula). |
| `data_fim_real` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de fim efetiva (pode ser nula). |
| `data_cadastro` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de cadastro do projeto na API. |
| `orgao_executor` | `category` | `VARCHAR` | Órgão responsável pela execução (achatado). |
| `fonte_recurso` | `category` | `VARCHAR` | Fonte de recurso (ex: FEDERAL, ESTADUAL). |
| `tipo_projeto` | `category` | `VARCHAR` | Tipo de projeto (ex: RODOVIA, EDUCAÇÃO). |
| `valor_total_previsto` | `float64` | `DOUBLE PRECISION` | Valor total de investimento previsto (agregado). |
| `ano_inicio_previsto` | `float64` | `DOUBLE PRECISION` | Ano de início previsto (derivado). |
| `mes_inicio_previsto` | `object` | `TEXT` | Mês/Ano de início previsto (convertido para string para compatibilidade). |
| `duracao_prevista_dias` | `float64` | `DOUBLE PRECISION` | Duração estimada do projeto em dias (derivado). |
| `is_concluido` | `bool` | `BOOLEAN` | Flag: `True` se o status indica conclusão (derivado). |
| `is_cancelado` | `bool` | `BOOLEAN` | Flag: `True` se o status indica cancelamento (derivado). |
| `is_em_execucao` | `bool` | `BOOLEAN` | Flag: `True` se o status indica execução (derivado). |
| `faixa_valor` | `category` | `VARCHAR` | Faixa de valor do investimento (derivado). |

#### Por que Usar um Banco de Dados?

O uso de um banco de dados relacional como o PostgreSQL é fundamental para:

1.  **Persistência e Durabilidade:** Armazenar os dados de forma segura e estruturada, garantindo que o dataset limpo persista além da execução do script.
2.  **Consultas e Performance:** Permitir consultas SQL complexas e eficientes, que são mais rápidas e escaláveis do que operações em DataFrames muito grandes.
3.  **Conformidade ETL:** Atender ao requisito de Carga (Load) do pipeline ETL em um ambiente de produção, separando a lógica de processamento da lógica de armazenamento.


## 6. Análise Quantitativa

### 6.1 Perguntas de Negócio

A análise exploratória visa responder às seguintes perguntas de negócio, fornecendo uma visão sobre a distribuição e o foco dos investimentos no Distrito Federal:


a. **Quais órgãos mais investem no DF?**  
   Análise do Top N órgãos executores por valor total previsto.

b. **Como se distribuem os projetos por tipo/categoria?**  
   Análise da proporção de projetos por `tipo_projeto`.

c. **Há evolução temporal nos investimentos?**  
   Análise da série temporal do valor de investimento previsto por ano.



### 6.2 - Análises Agregadas

Objetivo: Realizar os cálculos necessários para responder às perguntas de negócio.



In [69]:


# 1. Top N Órgãos Executores 
print("\n--- 1. Top 5 Órgãos Executores por Valor Total Previsto ---")
top_orgaos = (
    df_clean.groupby('orgao_executor', observed=True)['valor_total_previsto']
    .sum()
    .nlargest(5)
    .reset_index()
)
top_orgaos['percentual'] = (
    (top_orgaos['valor_total_previsto'] / df_clean['valor_total_previsto'].sum()) * 100
)
display(
    top_orgaos.style.format({
        'valor_total_previsto': 'R$ {:,.2f}',
        'percentual': '{:.2f}%'
    })
)
# 2. Distribuição por Tipo/Categoria 
print("\n--- 2. Distribuição de Projetos por Tipo/Categoria ---")
distribuicao_tipo = df_clean['tipo_projeto'].value_counts(normalize=True).mul(100).reset_index()
distribuicao_tipo.columns = ['tipo_projeto', 'percentual']
display(distribuicao_tipo.style.format({'percentual': '{:.2f}%'}))

# 3. Evolução Temporal 
print("\n--- 3. Evolução Temporal do Investimento Previsto por Ano ---")
investimento_anual = df_clean.groupby('ano_inicio_previsto')['valor_total_previsto'].sum().reset_index()
investimento_anual.columns = ['ano_inicio_previsto', 'valor_total_previsto']
display(investimento_anual.style.format({'valor_total_previsto': 'R$ {:,.2f}'}))




--- 1. Top 5 Órgãos Executores por Valor Total Previsto ---


,orgao_executor,valor_total_previsto,percentual
0,DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRANSPORTES,"R$ 3,820,711,205.56",48.79%
1,COMANDO DO EXÉRCITO,"R$ 973,834,040.61",12.44%
2,MINISTÉRIO DA INTEGRAÇÃO E DO DESENVOLVIMENTO REGIONAL,"R$ 560,500,027.00",7.16%
3,POLÍCIA MILITAR DO DISTRITO FEDERAL,"R$ 271,204,994.89",3.46%
4,DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRITO FEDERAL,"R$ 268,970,848.94",3.43%



--- 2. Distribuição de Projetos por Tipo/Categoria ---


,tipo_projeto,percentual
0,"INFRAESTRUTURA HÍDRICA, PORTOS, HIDROVIA",15.66%
1,ADMINISTRATIVO,15.37%
2,EDUCAÇÃO,14.51%
3,SEGURANÇA PÚBLICA,12.93%
4,RODOVIA,10.92%
5,SAÚDE,6.61%
6,ASSISTÊNCIA SOCIAL,5.46%
7,DESENVOLVIMENTO,4.45%
8,INFRAESTRUTURA URBANA E MOBILIDADE,3.74%
9,ESPORTE,1.87%



--- 3. Evolução Temporal do Investimento Previsto por Ano ---


,ano_inicio_previsto,valor_total_previsto
0,2003.000000,"R$ 54,496,950.34"
1,2007.000000,"R$ 723,884.88"
2,2008.000000,"R$ 106,571,942.23"
3,2010.000000,"R$ 91,759,187.00"
4,2011.000000,"R$ 18,059,833.76"
5,2012.000000,"R$ 143,258,163.13"
6,2013.000000,"R$ 123,092,637.83"
7,2014.000000,"R$ 106,447,092.20"
8,2015.000000,"R$ 44,535,697.02"
9,2016.000000,"R$ 11,780,477.64"


## 7. Visualizações

### 7.1 - Distribuição de Projetos por Faixa de Valor
Objetivo: Analisar a concentração de projetos em diferentes faixas de valor, permitindo entender a escala dos projetos predominantes.


In [84]:
import plotly.express as px
import os

# Garantir que o diretório de visualizações exista
os.makedirs("visualizacoes", exist_ok=True)

faixa_valor_counts = df_clean["faixa_valor"].value_counts().sort_index().reset_index()
faixa_valor_counts.columns = ["faixa_valor", "numero_projetos"]
fig3 = px.bar(faixa_valor_counts, x="faixa_valor", y="numero_projetos",
              title="1. Distribuição de Projetos por Faixa de Valor no DF",
              color="numero_projetos", color_continuous_scale=px.colors.sequential.Sunset)
fig3.update_layout(xaxis_title="Faixa de Valor", yaxis_title="Número de Projetos")
fig3.write_image("visualizacoes/03_faixa_valor.png")
fig3.show()

### 7.2 - Análise por Órgão

Objetivo: Visualizar a contribuição dos órgãos executores para o investimento total.



In [ ]:
# 1. Gráfico de Barras: Top 10 Órgãos por Valor
top_orgaos = (
    df_clean.groupby('orgao_executor', observed=True)['valor_total_previsto']
    .sum()
    .nlargest(10)
    .reset_index()
)

fig_bar = px.bar(
    top_orgaos,
    x='orgao_executor',
    y='valor_total_previsto',
    title='2. Top 10 Órgãos Executores por Valor de Investimento',
    labels={'orgao_executor': 'Órgão Executor', 'valor_total_previsto': 'Valor Total Previsto (R$)'},
    color='valor_total_previsto',
    color_continuous_scale=px.colors.sequential.Plasma
)

fig_bar.update_layout(xaxis={'categoryorder': 'total descending'})
fig_bar.write_image("visualizacoes/02_top_orgaos_barras.png")
fig_bar.show()

# 2. Gráfico de Pizza: Proporção do Total (Top 5 + Outros)
top_5_orgaos = df_clean['orgao_executor'].value_counts().nlargest(5).index.tolist()
df_pizza = df_clean.copy()
df_pizza['orgao_agregado'] = df_pizza['orgao_executor'].apply(lambda x: x if x in top_5_orgaos else 'OUTROS')

fig_pie = px.pie(
    df_pizza,
    names='orgao_agregado',
    values='valor_total_previsto',
    title='3. Proporção do Investimento por Órgão (Top 5 + Outros)',
    hole=.3
)

fig_pie.write_image("visualizacoes/03_proporcao_orgaos_pizza.png")
fig_pie.show()


### 7.3 - Análise Temporal

Objetivo: Visualizar a tendência de investimento ao longo do tempo.



In [ ]:
# Série temporal de investimentos agregados por ano
df_time = df_clean.dropna(subset=['ano_inicio_previsto']).copy()
investimento_anual = df_time.groupby('ano_inicio_previsto')['valor_total_previsto'].sum().reset_index()

fig_line = px.line(investimento_anual, x='ano_inicio_previsto', y='valor_total_previsto',
                   title='4. Série Temporal do Investimento Previsto por Ano',
                   labels={'ano_inicio_previsto': 'Ano de Início Previsto', 'valor_total_previsto': 'Valor Previsto (R$)'},
                   markers=True)

fig_line.update_layout(xaxis_tickformat = 'd') # Formato de ano
fig_line.write_image("visualizacoes/04_evolucao_temporal.png")
fig_line.show()

### 7.4 - Análises Categóricas

Objetivo: Comparar a distribuição de valores entre diferentes categorias (ex: Status).



In [83]:
status_counts = df_clean["status"].value_counts().reset_index()
status_counts.columns = ["status", "numero_projetos"]
fig5 = px.pie(status_counts, values="numero_projetos", names="status",
              title="5. Distribuição de Projetos por Status no DF",
              color_discrete_sequence=px.colors.sequential.RdBu)
fig5.update_layout(xaxis_title="Status", yaxis_title="Número de Projetos")
fig5.write_image("visualizacoes/05_status_projetos.png")
fig5.show()

In [82]:

def load_and_process_data(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        dados_brutos = json.load(f)

    df = pd.DataFrame(dados_brutos)

    # Função para extrair a descrição principal de uma lista de dicionários
    def extract_main_info(list_of_dicts, key='descricao'):
        if isinstance(list_of_dicts, list) and list_of_dicts:
            unique_descriptions = sorted(list(set([str(d.get(key, '')) for d in list_of_dicts if isinstance(d, dict)])))
            return '; '.join(unique_descriptions)
        return None

    # Aplicar a função para achatar a coluna 'tipos'
    df['tipos_principais'] = df['tipos'].apply(lambda x: extract_main_info(x, key='descricao'))
    
    return df
# Carregar e processar os dados
df_processed = load_and_process_data('data/raw/projetos_df_raw.json')

# Contar a frequência de cada tipo de projeto
tipos_contagem = df_processed['tipos_principais'].value_counts().reset_index()
tipos_contagem.columns = ['Tipo de Projeto', 'Número de Projetos']

# Criar o gráfico de barras interativo com Plotly Express
fig = px.bar(tipos_contagem, 
             x='Tipo de Projeto', 
             y='Número de Projetos', 
             title='6. Distribuição de Projetos por Tipo',
             labels={'Tipo de Projeto': 'Tipo de Projeto', 'Número de Projetos': 'Número de Projetos'},
             color='Número de Projetos', # Colore as barras com base no número de projetos
             color_continuous_scale=px.colors.sequential.Viridis)



# Salvar o gráfico como PNG
fig.write_image("visualizacoes/07_distribuicao_tipos_projeto.png")

# Exibir o gráfico
fig.show()


## 8. Análise Qualitativa
### 8.1 Padrões Identificados na Análise

Com base nas análises quantitativas e visuais, os seguintes padrões foram observados:

1.  **Perfil dos Projetos: Quantidade vs. Valor**
    * O Gráfico 1 (Distribuição de Projetos por Faixa de Valor) revela um padrão claro: a grande maioria dos projetos é de médio a baixo custo. A faixa de R$ 1 milhão a R$ 10 milhões concentra o maior número de projetos, seguida de perto pelos projetos de menos de R$ 100 mil. Isso indica que o volume de obras é composto, em sua maioria, por intervenções de menor escala.

2.  **Concentração de Investimentos**
    *  O Gráfico 2 (Top 10 Órgãos Executores por Valor de Investimento) e o Gráfico 3 (Proporção do Investimento por Órgão) mostram uma forte concentração de recursos. O DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRANSPORTES (DNIT) é, de longe, o maior investidor, responsável por quase metade de todo o valor previsto (48,8%).
    * Somando a participação do DNIT com a do Comando do Exército e a categoria "Outros", temos a maior parte dos investimentos. Isso demonstra que os projetos de maior impacto financeiro estão centralizados em poucos órgãos, principalmente federais, com foco em infraestrutura de grande porte.

3.  **Ciclos de Investimento**
    *   O Gráfico 4 (Série Temporal do Investimento Previsto por Ano) revela que o planejamento de investimentos não é constante. Observa-se um período de relativa estabilidade com valores mais baixos até 2016, seguido por um aumento e picos significativos em 2018, 2021 e 2023.
    * Essas flutuações podem estar ligadas a ciclos de planejamento governamental (como Planos Plurianuais), início de novos mandatos ou a aprovação de grandes projetos estratégicos que demandam um volume massivo de recursos em anos específicos.

4. **Status**
    * Os Gráfico 5 (Distribuição de Projetos por Status) apresenta um dos insights mais críticos da análise. Uma esmagadora maioria dos projetos (76,4%, ou mais de 500 projetos) está no status "CADASTRADA". Em contraste, apenas 11,2% estão "EM EXECUÇÃO" e 8,48% estão "CONCLUÍDA".
    * Isso configura uma verdadeira disparidade, onde um grande volume de projetos é planejado, mas uma fração muito menor avança para a execução e conclusão.

 5. **Tipos de Projeto**
    * O Gráfico 6 (Distribuição de Projetos por Tipo) mostra a variedade de áreas que recebem investimentos. Os tipos de projeto com maior número de iniciativas são Infraestrutura Hídrica, Portos, Hidrovia, Educação, Administrativo e Rodovia.
    * Isso indica que, em termos de quantidade de projetos, há um esforço distribuído em áreas essenciais para o desenvolvimento social e de infraestrutura.

### 8.2 Formulação de Hipóteses

1. **Burocracia Orçamentária:**
   * A alta concentração de projetos no status "CADASTRADA" pode ser resultado de processos burocráticos lentos (licenciamento, aprovações) ou da dependência de liberação de verbas orçamentárias, que podem não ocorrer no ritmo do planejamento. Os picos de investimento vistos no gráfico temporal podem corresponder a anos em que houve maior liberação de recursos.

2. **Planejamento de Longo Prazo vs. Execução Imediata:**
   * Muitos projetos cadastrados podem fazer parte de um planejamento estratégico de longo prazo, sem previsão de execução imediata. Eles funcionam como um "banco de projetos", aguardando a disponibilidade de recursos ou a janela de oportunidade política e administrativa para serem iniciados.

3. **Foco Estratégico em Infraestrutura:**
   * A dominância do DNIT e de projetos rodoviários (em valor e quantidade, respectivamente) sugere uma prioridade estratégica do governo federal em melhorar a infraestrutura de transportes do Distrito Federal, que serve como um nó logístico central para o país.

## 9. Conclusão

Este projeto demonstrou a construção de um pipeline ETL (Extração, Transformação e Carga) completo, consumindo dados da API ObrasGov.br, tratando estruturas complexas (listas aninhadas) e persistindo o resultado em um banco de dados PostgreSQL.

